# 01 — Data Exploration and Table 3 Reconciliation
**QM640 Capstone — Pollution-Prevention Actions and Next-Year Toxic Release Reduction**

This notebook performs exploratory data analysis on the EPA TRI Basic Plus files
(File 1A and File 2A) for every reporting year in scope, 2020-2024, and reproduces
Table 3 of the synopsis by applying the same filters used in `src/data_prep.py`:
Form R records only, primary NAICS beginning with 31/32/33 (manufacturing), then
consolidating duplicate facility-chemical-year records.

**Reporting year 2024 is the held-out test set** (the 2023->2024 transition) and is
flagged throughout -- it is never used during model development.

Run this notebook after downloading the raw files per `data/README.md` into `data/raw/`.


In [ ]:
import sys
sys.path.append("../src")
from data_prep import load_filtered_1a, consolidate, YEARS
from pathlib import Path

DATA_DIR = Path("../data/raw")

## Raw file shape, by reporting year

`df.shape` on the raw File 1A / File 2A files, straight off disk, before any filtering.
This is the same view shown in Appendix E, Figures E2-E6 of the synopsis.

In [ ]:
import pandas as pd

raw_shapes = {}
for year in YEARS:
    df1, _, _ = load_filtered_1a(DATA_DIR, year)
    df2 = pd.read_csv(DATA_DIR / f"us_{year}" / f"US_2a_{year}.txt", sep="\t",
                       index_col=False, encoding="latin1", on_bad_lines="skip",
                       engine="c", low_memory=False)
    raw_shapes[year] = (df1.shape, df2.shape)
    tag = "  <-- TEST/HOLDOUT" if year == 2024 else ""
    print(f"{year}: File 1A {df1.shape[0]:,} rows x {df1.shape[1]} cols | "
          f"File 2A {df2.shape[0]:,} rows x {df2.shape[1]} cols{tag}")

## Reconciling raw counts with Table 3

Table 3 of the synopsis reports counts **after** the Form R filter, the manufacturing
NAICS filter, and duplicate consolidation -- not the raw row count above. This cell
walks each year through every stage and must match Table 3 exactly (see Appendix E,
Figure E1).

In [ ]:
print(f"{'Year':>6} {'Raw rows':>12} {'Form R':>12} {'+ Manufacturing':>16} {'Consolidated':>14}")
consolidated_by_year = {}
for year in YEARS:
    raw, form_r, manufacturing = load_filtered_1a(DATA_DIR, year)
    consolidated = consolidate(manufacturing)
    consolidated_by_year[year] = consolidated
    print(f"{year:>6} {len(raw):>12,} {len(form_r):>12,} {len(manufacturing):>16,} "
          f"{len(consolidated):>14,}")

# Expected from Table 3 (synopsis): 56,462 / 56,219 / 56,155 / 55,349 / 54,115
assert consolidated_by_year[2020].shape[0] == 56_462
assert consolidated_by_year[2021].shape[0] == 56_219
assert consolidated_by_year[2022].shape[0] == 56_155
assert consolidated_by_year[2023].shape[0] == 55_349
assert consolidated_by_year[2024].shape[0] == 54_115
print("\nAll five years match Table 3 exactly.")

## Dataset preview (consolidated, manufacturing Form R records)

In [ ]:
cols = ["9. TRIFD", "10. FACILITY NAME", "12. FACILITY CITY", "14. FACILITY STATE",
        "41. PRIMARY NAICS CODE", "81. CHEMICAL NAME", "85. UNIT OF MEASURE"]
consolidated_by_year[2020][[c for c in cols if c in consolidated_by_year[2020].columns]].head(6)

## Column types and memory footprint, by year (File 1A, raw)

In [ ]:
for year in YEARS:
    df1, _, _ = load_filtered_1a(DATA_DIR, year)
    dtypes = df1.dtypes.value_counts()
    tag = "  <-- TEST/HOLDOUT" if year == 2024 else ""
    print(f"{year}: {df1.shape[0]:,} rows | {dict(dtypes)} | "
          f"{df1.memory_usage(deep=True).sum()/1e6:.1f} MB{tag}")

## Missing values (top columns, File 1A, raw), by year

In [ ]:
for year in YEARS:
    df1, _, _ = load_filtered_1a(DATA_DIR, year)
    miss = df1.isna().sum().sort_values(ascending=False)
    print(f"--- {year} ---")
    print(miss[miss > 0].head(5))
    print()

## Reported pollution-prevention action rate by year (matches Table 3)

Reported-action rate = consolidated facility-chemical-year records with at least one
valid Source Reduction Activity Code (File 2A) / all consolidated facility-chemical-year
records that year.

In [ ]:
import matplotlib.pyplot as plt

years = [2020, 2021, 2022, 2023, 2024]
num = [2304, 3154, 3302, 3293, 3020]
den = [consolidated_by_year[y].shape[0] for y in years]
rate = [n / d * 100 for n, d in zip(num, den)]

colors = ["#2e7d32"] * 4 + ["#c0392b"]
plt.figure(figsize=(7, 4))
plt.bar([str(y) for y in years], rate, color=colors)
plt.ylabel("Reported-action rate (%)")
plt.title("Reported pollution-prevention action rate by year (2024 = test/holdout)")
plt.show()

## Next steps

- Join consolidated File 1A with File 2A within each year on `DOCUMENT CONTROL NUMBER`
  (see `data_prep.load_year()`).
- Match consecutive years to construct `REDUCTION_10` (Appendix A, Table A4 of the synopsis).
- Proceed to RQ1-RQ4 analysis per Appendix C and D of the synopsis, holding out the
  2023->2024 transition until final testing.